In [1]:
import polars as pl
import sys
from pathlib import Path
import os
ROOT = Path.cwd().resolve().parent   # process/ 的上一级
sys.path.insert(0, str(ROOT)) # 把临时目查到如列表的第0位（最高级）让解释器从这个目录开始找模块
from candidate.recall_fun import recall_popularity,recall_itemcf,recall_repurchase_decay,recall_w2vec,build_itemcf,train_w2vec

In [2]:
articles_path='../data/articles.parquet'
transaction_path='../data/transactions.parquet'
articles=pl.read_parquet(articles_path)
transactions=pl.read_parquet(transaction_path)

In [3]:
DAY = 86400
WEEK = 7 * DAY

max_time = transactions.select(pl.max("time")).item()

valid_start = max_time - WEEK
train_start = valid_start - 12 * WEEK   # 12 周

train = transactions.filter(
    (pl.col("time") >= train_start) &
    (pl.col("time") < valid_start)
)

valid= transactions.filter(
    pl.col("time") >= valid_start
)


In [5]:
def preserve_popularity_result(data,week=1):
    for w in range(1,week+1):
        res=recall_popularity(data,window_days=7*week)
        res.write_parquet(f'recall_popularity_week{w}.parquet')

In [6]:
preserve_popularity_result(train,4)

100%|██████████| 508758/508758 [00:01<00:00, 277728.12it/s]


In [13]:
res=recall_repurchase_decay(train)
res.write_parquet('recall_repurchase.parquet')

100%|██████████| 508758/508758 [01:58<00:00, 4310.71it/s]


In [5]:
item_sim=build_itemcf(train)
res=recall_itemcf(train,item_sim)
res.write_parquet('itemcf_recall.parquet')

100%|██████████| 508758/508758 [04:05<00:00, 2070.45it/s]


In [4]:
res=recall_w2vec(train,topk=50,hist_len=10,is_valid=True,model_path='')
res.write_parquet('recall_w2vec.parquet')

100%|██████████| 508758/508758 [02:19<00:00, 3651.76it/s]
